# Persian STT Benchmark — FLEURS fa_ir Test Split

Benchmarks **15 local models** across five families on the full FLEURS fa_ir test split (871 clips).

| Family | Models |
| --- | --- |
| Whisper (stock) | tiny · base · small · medium · large-v3 · large-v3-turbo |
| Whisper (fine-tuned fa) | whisper-small-fa · whisper-large-v3-fa · whisper-large-fa-v1 · whisper-persian-v4 |
| Wav2Vec2 XLSR | wav2vec2-large-xlsr-53-persian |
| Meta SeamlessM4T | hf-seamless-m4t-medium · seamless-m4t-v2-large |
| Meta MMS | mms-1b-all · mms-1b-fl102 |

**Metrics (computed per model):**
- **Accuracy:** Corpus WER · Corpus CER · chrF · BERTScore (ParsBERT) · semantic similarity
- **Reliability:** SER · WER P50/P90/P95 · %perfect · %catastrophic · sub/ins/del rates · empty-output rate
- **Persian-specific failures:** hallucination ratio · script contamination · repetition score · punctuation F1
- **Speed / footprint:** avg inference time · RTF · throughput (×real-time) · peak VRAM · WER by audio-duration bucket

**Dataset loading — two modes (automatic):**
- **Fast (subsequent sessions):** Attach the `fleurs-fa-ir-test` Kaggle dataset in Settings → Add data. The notebook loads from disk instantly — no internet required.
- **First run only:** If the dataset is not attached, the notebook downloads FLEURS fa_ir from HuggingFace (~844 MB) and saves it to `/kaggle/working/fleurs_fa_ir_test`. After the run, publish that folder as a private Kaggle dataset named `fleurs-fa-ir-test` so all future sessions use it.

Install required Python packages.

In [ ]:
!pip install -q jiwer hazm sacrebleu sentencepiece
!pip install -q --no-deps bert-score sentence-transformers

Import libraries and verify GPU is available.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  IMPORTS
# ════════════════════════════════════════════════════════════════
import os, gc, re, time, io, warnings
from collections import Counter
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from datasets import load_dataset, load_from_disk, Audio
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    SeamlessM4TModel, SeamlessM4Tv2Model, AutoProcessor,
)
from jiwer import wer, cer, process_words
from sacrebleu.metrics import CHRF
import hazm

# ════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ════════════════════════════════════════════════════════════════
warnings.filterwarnings("ignore")

RESULTS_DIR  = "/kaggle/working"
DATASET_PATH = "/kaggle/input/Fleurs_fa_ir_test/fleurs_fa_ir_test"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Load FLEURS fa_ir test split — from attached Kaggle dataset if available, otherwise download from HuggingFace and save for future sessions.

In [ ]:
import os
print(os.listdir('/kaggle/input'))
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    if root.count('/') > 3:
        break

In [ ]:
# ════════════════════════════════════════════════════════════════
#  LOAD DATASET
# ════════════════════════════════════════════════════════════════
if os.path.exists(DATASET_PATH):
    print(f"Loading FLEURS fa_ir from attached dataset: {DATASET_PATH}")
    ds = load_from_disk(DATASET_PATH)
else:
    print(f"Kaggle dataset not attached — downloading from HuggingFace (~844 MB)...")
    print("Enable Internet in notebook Settings if this hangs.")
    ds = load_dataset("google/fleurs", "fa_ir", split="test")
    save_path = "/kaggle/working/fleurs_fa_ir_test"
    print(f"Saving to {save_path} for future sessions...")
    ds.save_to_disk(save_path)
    print("Done. After this run: Output tab → publish 'fleurs_fa_ir_test' as a")
    print("private Kaggle dataset named 'fleurs-fa-ir-test', then attach it next session.")

print(f"\nClips   : {len(ds)}")
print(f"Columns : {ds.column_names}")

Decode audio and build the clips list that all models will iterate over.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  BUILD CLIPS LIST
# ════════════════════════════════════════════════════════════════
ds_raw = ds.cast_column("audio", Audio(decode=False))

clips = []
for i, row in enumerate(ds_raw):
    a = row["audio"]
    if a.get("bytes"):
        arr, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
    else:
        arr, sr = sf.read(a["path"], dtype="float32")
    sr = int(sr)

    if arr.ndim > 1:
        arr = arr.mean(axis=1).astype(np.float32)

    transcript = (
        row.get("transcription")
        or row.get("raw_transcription")
        or ""
    ).strip()

    clips.append({
        "clip_id"     : i,
        "audio_array" : arr,
        "sample_rate" : sr,
        "duration_sec": len(arr) / sr,
        "reference"   : transcript,
    })

total_min = sum(c["duration_sec"] for c in clips) / 60
print(f"Parsed {len(clips)} clips — {total_min:.1f} minutes of audio")
print(f"Sample rate: {clips[0]['sample_rate']} Hz")

Define the Persian text normalizer, helper scorers, the `evaluate_model` runner, and the GPU cleanup helper.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  SHARED UTILITIES
# ════════════════════════════════════════════════════════════════
_hazm_norm   = hazm.Normalizer()
_chrf_metric = CHRF()

_PERSIAN_RANGES = [('؀', 'ۿ'), ('ﭐ', '﷿'), ('ﹰ', '﻿')]

def _is_persian_char(c):
    return any(lo <= c <= hi for lo, hi in _PERSIAN_RANGES)

def normalize_persian(text: str) -> str:
    """Hazm-normalize Persian text and strip punctuation."""
    text = _hazm_norm.normalize(text)
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _script_contamination(text: str) -> float:
    """Fraction of alphabetic characters that are NOT Persian/Arabic script."""
    alpha = [c for c in text if c.isalpha()]
    if not alpha:
        return 0.0
    non_persian = sum(1 for c in alpha if not _is_persian_char(c))
    return non_persian / len(alpha)

def _repetition_score(text: str, n: int = 4) -> float:
    """Fraction of 4-grams that are repeated (Whisper hallucination signature)."""
    words = text.split()
    if len(words) < n + 1:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(v - 1 for v in counts.values() if v > 1)
    return repeated / len(ngrams)

def _punct_f1(ref: str, hyp: str) -> float:
    """F1 for Persian punctuation marks (.,،؟!؛:)."""
    PUNCTS = set('.,،؟!؛:')
    r = [c for c in ref if c in PUNCTS]
    h = [c for c in hyp if c in PUNCTS]
    if not r and not h:
        return 1.0
    if not r or not h:
        return 0.0
    rc, hc = Counter(r), Counter(h)
    match = sum((rc & hc).values())
    prec  = match / len(h)
    rec   = match / len(r)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def free_gpu():
    """Release GPU memory after each model."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_model(transcribe_fn, clips, model_name):
    """
    Run transcribe_fn on every clip, compute all metrics, save CSV.

    Per-clip columns:
        clip_id, duration_sec, reference, hypothesis,
        wer, cer, chrf, hallucination_ratio, is_empty,
        script_contamination, repetition_score, punct_f1,
        inference_time_sec, rtf

    SUMMARY row (all of the above as aggregates) plus:
        ser, wer_p50, wer_p90, wer_p95, pct_perfect, pct_catastrophic,
        sub_rate, ins_rate, del_rate,
        throughput_audio_hrs_per_hr, peak_vram_gb,
        wer_lt5s, wer_5to15s, wer_gt15s

    BERTScore and semantic similarity are computed in the master results cell.
    """
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    print(f"\nEvaluating: {model_name}  ({len(clips)} clips)")
    records = []

    for i, clip in enumerate(clips):
        t0 = time.perf_counter()
        try:
            hyp = transcribe_fn(clip["audio_array"], clip["sample_rate"])
        except Exception as exc:
            hyp = ""
            print(f"  [WARN] clip {clip['clip_id']} failed: {exc}")
        elapsed = time.perf_counter() - t0

        ref   = clip["reference"]
        ref_n = normalize_persian(ref)
        hyp_n = normalize_persian(hyp)

        try:
            clip_wer = wer(ref_n, hyp_n) if ref_n else float("nan")
            clip_cer = cer(ref_n, hyp_n) if ref_n else float("nan")
        except Exception:
            clip_wer = clip_cer = float("nan")

        try:
            clip_chrf = _chrf_metric.sentence_score(hyp_n, [ref_n]).score if ref_n else float("nan")
        except Exception:
            clip_chrf = float("nan")

        ref_words = len(ref_n.split()) if ref_n else 0
        hyp_words = len(hyp_n.split())

        records.append({
            "clip_id"             : clip["clip_id"],
            "duration_sec"        : round(clip["duration_sec"], 3),
            "reference"           : ref,
            "hypothesis"          : hyp,
            "wer"                 : round(clip_wer, 4),
            "cer"                 : round(clip_cer, 4),
            "chrf"                : round(clip_chrf, 4) if not np.isnan(clip_chrf) else float("nan"),
            "hallucination_ratio" : round(hyp_words / max(ref_words, 1), 4),
            "is_empty"            : int(hyp.strip() == ""),
            "script_contamination": round(_script_contamination(hyp), 4),
            "repetition_score"    : round(_repetition_score(hyp), 4),
            "punct_f1"            : round(_punct_f1(ref, hyp), 4),
            "inference_time_sec"  : round(elapsed, 4),
            "rtf"                 : round(elapsed / clip["duration_sec"], 4),
        })

        if (i + 1) % 100 == 0:
            avg_wer = np.nanmean([r["wer"] for r in records])
            print(f"  [{i + 1}/{len(clips)}]  running avg WER = {avg_wer:.4f}")

    df    = pd.DataFrame(records)
    valid = df[df["reference"].str.strip() != ""]

    # ── CORPUS METRICS ────────────────────────────────────────────────────────
    ref_list = [normalize_persian(r) for r in valid["reference"]]
    hyp_list = [normalize_persian(h) for h in valid["hypothesis"]]
    ref_corp  = " ".join(ref_list)
    hyp_corp  = " ".join(hyp_list)

    corp_wer = round(wer(ref_corp, hyp_corp), 4) if ref_corp else float("nan")
    corp_cer = round(cer(ref_corp, hyp_corp), 4) if ref_corp else float("nan")

    try:
        corp_chrf = round(_chrf_metric.corpus_score(hyp_list, [ref_list]).score, 4)
    except Exception:
        corp_chrf = float("nan")

    # ── ERROR BREAKDOWN ───────────────────────────────────────────────────────
    try:
        pw              = process_words(ref_list, hyp_list)
        total_ref_words = pw.hits + pw.substitutions + pw.deletions
        sub_rate = round(pw.substitutions / total_ref_words, 4) if total_ref_words else float("nan")
        ins_rate = round(pw.insertions    / total_ref_words, 4) if total_ref_words else float("nan")
        del_rate = round(pw.deletions     / total_ref_words, 4) if total_ref_words else float("nan")
    except Exception:
        sub_rate = ins_rate = del_rate = float("nan")

    # ── WER DISTRIBUTION ──────────────────────────────────────────────────────
    wer_vals         = df["wer"].dropna().values
    wer_p50          = round(float(np.percentile(wer_vals, 50)),   4) if len(wer_vals) else float("nan")
    wer_p90          = round(float(np.percentile(wer_vals, 90)),   4) if len(wer_vals) else float("nan")
    wer_p95          = round(float(np.percentile(wer_vals, 95)),   4) if len(wer_vals) else float("nan")
    pct_perfect      = round(float(np.mean(wer_vals == 0)),        4) if len(wer_vals) else float("nan")
    pct_catastrophic = round(float(np.mean(wer_vals > 0.5)),       4) if len(wer_vals) else float("nan")
    ser              = round(float(np.mean(wer_vals > 0)),         4) if len(wer_vals) else float("nan")

    # ── AVERAGES & DEPLOYMENT METRICS ─────────────────────────────────────────
    empty_rate = round(df["is_empty"].mean(), 4)
    avg_inf    = round(df["inference_time_sec"].mean(), 4)
    avg_rtf    = round(df["rtf"].mean(), 4)
    throughput = round(1.0 / avg_rtf, 4) if avg_rtf > 0 else float("nan")

    peak_vram = round(torch.cuda.max_memory_allocated() / 1e9, 3) if torch.cuda.is_available() else 0.0

    # ── WER BY DURATION BUCKET ────────────────────────────────────────────────
    def _bucket_wer(lo, hi):
        mask = (df["duration_sec"] >= lo) & (df["duration_sec"] < hi)
        vals = df.loc[mask, "wer"].dropna()
        return round(float(np.nanmean(vals)), 4) if len(vals) else float("nan")

    wer_lt5s   = _bucket_wer(0,  5)
    wer_5to15s = _bucket_wer(5,  15)
    wer_gt15s  = _bucket_wer(15, 9999)

    # ── SUMMARY ROW ───────────────────────────────────────────────────────────
    summary = {
        "clip_id"                    : "SUMMARY",
        "duration_sec"               : round(df["duration_sec"].sum(), 2),
        "reference"                  : "",
        "hypothesis"                 : "",
        "wer"                        : corp_wer,
        "cer"                        : corp_cer,
        "chrf"                       : corp_chrf,
        "hallucination_ratio"        : round(df["hallucination_ratio"].mean(), 4),
        "is_empty"                   : empty_rate,
        "script_contamination"       : round(df["script_contamination"].mean(), 4),
        "repetition_score"           : round(df["repetition_score"].mean(), 4),
        "punct_f1"                   : round(df["punct_f1"].mean(), 4),
        "inference_time_sec"         : avg_inf,
        "rtf"                        : avg_rtf,
        "ser"                        : ser,
        "wer_p50"                    : wer_p50,
        "wer_p90"                    : wer_p90,
        "wer_p95"                    : wer_p95,
        "pct_perfect"                : pct_perfect,
        "pct_catastrophic"           : pct_catastrophic,
        "sub_rate"                   : sub_rate,
        "ins_rate"                   : ins_rate,
        "del_rate"                   : del_rate,
        "throughput_audio_hrs_per_hr": throughput,
        "peak_vram_gb"               : peak_vram,
        "wer_lt5s"                   : wer_lt5s,
        "wer_5to15s"                 : wer_5to15s,
        "wer_gt15s"                  : wer_gt15s,
    }

    df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)

    safe     = model_name.replace("/", "__")
    csv_path = os.path.join(RESULTS_DIR, f"results__{safe}.csv")
    df.to_csv(csv_path, index=False)

    sep = "=" * 66
    print(f"\n{sep}")
    print(f"  {model_name}")
    print(f"  Corpus WER         : {corp_wer:.4f}  (P50={wer_p50:.3f}  P90={wer_p90:.3f}  P95={wer_p95:.3f})")
    print(f"  Corpus CER         : {corp_cer:.4f}")
    print(f"  Corpus chrF        : {corp_chrf:.2f}")
    print(f"  SER                : {ser:.4f}  (perfect={pct_perfect:.1%}  catastrophic={pct_catastrophic:.1%})")
    print(f"  Sub / Ins / Del    : {sub_rate:.4f} / {ins_rate:.4f} / {del_rate:.4f}")
    print(f"  Hallucination ratio: {summary['hallucination_ratio']:.4f}")
    print(f"  Script contam.     : {summary['script_contamination']:.4f}")
    print(f"  Repetition score   : {summary['repetition_score']:.4f}")
    print(f"  Empty output rate  : {empty_rate:.4f}")
    print(f"  Punct F1           : {summary['punct_f1']:.4f}")
    print(f"  Avg RTF            : {avg_rtf:.4f}  (throughput = {throughput:.2f}× real-time)")
    print(f"  Peak VRAM          : {peak_vram:.2f} GB")
    print(f"  WER by duration    : <5s={wer_lt5s:.4f}  5–15s={wer_5to15s:.4f}  >15s={wer_gt15s:.4f}")
    print(f"  Saved → {csv_path}")
    print(sep)
    return df


---
## 1 — Whisper (stock weights)
Six size variants: tiny · base · small · medium · large-v3 · large-v3-turbo.  
All use `WhisperForConditionalGeneration` + `WhisperProcessor`, fp16 on GPU.

### 1.1 — whisper-tiny

Load `whisper-tiny` weights and processor from HuggingFace.

In [ ]:
MODEL_ID    = "openai/whisper-tiny"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-tiny`, compute metrics, and save per-clip results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_01 = evaluate_model(_t, clips, "openai/whisper-tiny")
del w_model, w_proc; free_gpu()

### 1.2 — whisper-base

Load `whisper-base` weights and processor.

In [ ]:
MODEL_ID    = "openai/whisper-base"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-base`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_02 = evaluate_model(_t, clips, "openai/whisper-base")
del w_model, w_proc; free_gpu()

### 1.3 — whisper-small

Load `whisper-small` weights and processor.

In [ ]:
MODEL_ID    = "openai/whisper-small"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-small`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_03 = evaluate_model(_t, clips, "openai/whisper-small")
del w_model, w_proc; free_gpu()

### 1.4 — whisper-medium

Load `whisper-medium` weights and processor.

In [ ]:
MODEL_ID    = "openai/whisper-medium"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-medium`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_04 = evaluate_model(_t, clips, "openai/whisper-medium")
del w_model, w_proc; free_gpu()

### 1.5 — whisper-large-v3

Load `whisper-large-v3` weights and processor.

In [ ]:
MODEL_ID    = "openai/whisper-large-v3"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-large-v3`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_05 = evaluate_model(_t, clips, "openai/whisper-large-v3")
del w_model, w_proc; free_gpu()

### 1.6 — whisper-large-v3-turbo

Load `whisper-large-v3-turbo` weights and processor.

In [ ]:
MODEL_ID    = "openai/whisper-large-v3-turbo"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-large-v3-turbo`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat, language="fa", task="transcribe")
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_06 = evaluate_model(_t, clips, "openai/whisper-large-v3-turbo")
del w_model, w_proc; free_gpu()

---
## 2 — Whisper fine-tuned Persian
Community fine-tunes on top of stock Whisper weights, trained on Persian data.
Same inference code as stock Whisper — only the weights differ.

### 2.1 — whisper-small-fa

Load `whisper-small-fa` weights and processor.

In [ ]:
MODEL_ID    = "steja/whisper-small-persian"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
w_model.generation_config.forced_decoder_ids = None
print("Ready.")

Transcribe all clips with `whisper-small-fa`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_07 = evaluate_model(_t, clips, "steja/whisper-small-persian")
del w_model, w_proc; free_gpu()

### 2.2 — whisper-large-v3-fa

Load `whisper-large-v3-fa` weights and processor.

In [ ]:
MODEL_ID    = "MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `whisper-large-v3-fa`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_09 = evaluate_model(_t, clips, "MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch")
del w_model, w_proc; free_gpu()

---
## 3 — Wav2Vec2 XLSR (Persian fine-tune)
CTC-based model — no beam search, no language model. Output has no punctuation.
Requires 16 kHz mono audio.

### 3.1 — wav2vec2-large-xlsr-53-persian

Load Wav2Vec2 model and processor from HuggingFace.

In [ ]:
MODEL_ID    = "jonatasgrosman/wav2vec2-large-xlsr-53-persian"
print(f"Loading {MODEL_ID} ...")
w2v_proc  = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = (
    Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with Wav2Vec2, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = w2v_proc(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = w2v_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return w2v_proc.batch_decode(pred_ids)[0]

df_10 = evaluate_model(_t, clips, "jonatasgrosman/wav2vec2-large-xlsr-53-persian")
del w2v_model, w2v_proc; free_gpu()

---
## 4 — Meta SeamlessM4T
Speech-to-text via the multilingual SeamlessM4T models.  
Persian language code: `pes` (ISO 639-3).

### 4.1 — hf-seamless-m4t-medium

Load `hf-seamless-m4t-medium` model and processor (the `hf-` repo is the transformers-compatible one).

In [ ]:
MODEL_ID   = "facebook/hf-seamless-m4t-medium"
print(f"Loading {MODEL_ID} ...")
sm_proc  = AutoProcessor.from_pretrained(MODEL_ID)
sm_model = (
    SeamlessM4TModel.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `seamless-m4t-medium`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = sm_proc(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = sm_model.generate(**inputs, tgt_lang="pes", generate_speech=False)
    seqs = out.sequences if hasattr(out, "sequences") else out
    return sm_proc.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]

df_11 = evaluate_model(_t, clips, "facebook/hf-seamless-m4t-medium")
del sm_model, sm_proc; free_gpu()

### 4.2 — seamless-m4t-v2-large

Load `seamless-m4t-v2-large` model and processor.

In [ ]:
MODEL_ID   = "facebook/seamless-m4t-v2-large"
print(f"Loading {MODEL_ID} ...")
sm_proc  = AutoProcessor.from_pretrained(MODEL_ID)
sm_model = (
    SeamlessM4Tv2Model.from_pretrained(MODEL_ID)
    .to(device).eval()
)
print("Ready.")

Transcribe all clips with `seamless-m4t-v2-large`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = sm_proc(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = sm_model.generate(**inputs, tgt_lang="pes", generate_speech=False)
    seqs = out.sequences if hasattr(out, "sequences") else out
    return sm_proc.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]

df_12 = evaluate_model(_t, clips, "facebook/seamless-m4t-v2-large")
del sm_model, sm_proc; free_gpu()

---
## 5 — Meta MMS (Massively Multilingual Speech)
CTC-based, same architecture as Wav2Vec2 but with per-language adapters.

**Important:** MMS ASR adapters only ship on the **1B** checkpoints (there is no
300M ASR model — `mms-300m` is a base model for fine-tuning only). MMS uses the
ISO 639-3 code **`fas`** for Persian (note: SeamlessM4T uses `pes` instead).

- `mms-1b-all` — 1162 languages, zero-shot on FLEURS.
- `mms-1b-fl102` — fine-tuned on FLEURS-102, so its score here is **in-domain (optimistic)**.

### 5.1 — mms-1b-fl102

Load `mms-1b-fl102` and activate the Persian (`fas`) language adapter.

In [ ]:
MODEL_ID = "facebook/mms-1b-fl102"
MMS_LANG = "fas"
print(f"Loading {MODEL_ID} ...")
mms_proc  = AutoProcessor.from_pretrained(MODEL_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()
mms_proc.tokenizer.set_target_lang(MMS_LANG)
mms_model.load_adapter(MMS_LANG)
print(f"Ready (Persian adapter: {MMS_LANG}).")

Transcribe all clips with `mms-1b-fl102`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = mms_proc(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = mms_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return mms_proc.decode(pred_ids[0])

df_13 = evaluate_model(_t, clips, "facebook/mms-1b-fl102")
del mms_model, mms_proc; free_gpu()

### 5.2 — mms-1b-all

Load `mms-1b-all` and activate the Persian (`fas`) language adapter.

In [ ]:
MODEL_ID = "facebook/mms-1b-all"
MMS_LANG = "fas"
print(f"Loading {MODEL_ID} ...")
mms_proc  = AutoProcessor.from_pretrained(MODEL_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device).eval()
mms_proc.tokenizer.set_target_lang(MMS_LANG)
mms_model.load_adapter(MMS_LANG)
print(f"Ready (Persian adapter: {MMS_LANG}).")

Transcribe all clips with `mms-1b-all`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    inputs = mms_proc(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = mms_model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return mms_proc.decode(pred_ids[0])

df_14 = evaluate_model(_t, clips, "facebook/mms-1b-all")
del mms_model, mms_proc; free_gpu()

---
## 6 — Additional Persian Whisper fine-tunes
Two recent community fine-tunes added after the original 14-model run.
Same inference path as the other Whisper fine-tunes — only the weights differ.

### 6.1 — whisper-large-fa-v1

Load `vhdm/whisper-large-fa-v1` weights and processor (fine-tuned on top of `whisper-large-v3-turbo`).

In [ ]:
MODEL_ID    = "vhdm/whisper-large-fa-v1"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
w_model.generation_config.forced_decoder_ids = None
print("Ready.")

Transcribe all clips with `whisper-large-fa-v1`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_15 = evaluate_model(_t, clips, "vhdm/whisper-large-fa-v1")
del w_model, w_proc; free_gpu()

### 6.2 — whisper-persian-v4

Load `nezamisafa/whisper-persian-v4` weights and processor (fine-tuned on top of `whisper-large-v3`).

In [ ]:
MODEL_ID    = "nezamisafa/whisper-persian-v4"
_dtype      = torch.float16 if device == "cuda" else torch.float32
print(f"Loading {MODEL_ID} ...")
w_proc  = WhisperProcessor.from_pretrained(MODEL_ID)
w_model = (
    WhisperForConditionalGeneration
    .from_pretrained(MODEL_ID, torch_dtype=_dtype)
    .to(device).eval()
)
w_model.generation_config.forced_decoder_ids = None
print("Ready.")

Transcribe all clips with `whisper-persian-v4`, compute metrics, and save results to CSV.

In [ ]:
def _t(audio, sr):
    if sr != 16000:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, 16000
        ).squeeze(0).numpy()
    feat = w_proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    if device == "cuda":
        feat = feat.half()
    with torch.no_grad():
        ids = w_model.generate(feat)
    return w_proc.batch_decode(ids, skip_special_tokens=True)[0]

df_16 = evaluate_model(_t, clips, "nezamisafa/whisper-persian-v4")
del w_model, w_proc; free_gpu()

---
## Master Results
Merge all per-model CSVs into a single summary table sorted by Corpus WER.

Read all per-model CSVs, compute BERTScore and semantic similarity on the text outputs, merge everything into one master table sorted by corpus WER.

In [ ]:
import glob
from bert_score import BERTScorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# ════════════════════════════════════════════════════════════════
#  SETUP
# ════════════════════════════════════════════════════════════════
csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "results__*.csv")))
print(f"Found {len(csv_files)} result file(s).\n")

BERT_MODEL  = "HooshvareLab/bert-base-parsbert-uncased"
BERT_LAYERS = 9

print(f"Loading BERTScore model ({BERT_MODEL})...")
bert_scorer = BERTScorer(model_type=BERT_MODEL, num_layers=BERT_LAYERS, device=device, idf=False)
bert_scorer._tokenizer.model_max_length = 512
print("BERTScore ready.")

print(f"Loading sentence encoder (paraphrase-multilingual-MiniLM-L12-v2)...")
sem_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("Sentence encoder ready.\n")

# ════════════════════════════════════════════════════════════════
#  PER-MODEL SCORING
# ════════════════════════════════════════════════════════════════
rows = []
for path in csv_files:
    df_tmp   = pd.read_csv(path)
    summary  = df_tmp[df_tmp["clip_id"] == "SUMMARY"].iloc[0]
    per_clip = df_tmp[df_tmp["clip_id"] != "SUMMARY"].copy()

    fname      = os.path.basename(path)
    model_name = fname[len("results__"):-len(".csv")].replace("__", "/", 1)
    print(f"── {model_name}")

    valid = per_clip[per_clip["reference"].fillna("").str.strip() != ""].copy()
    refs  = valid["reference"].fillna("").astype(str).tolist()
    hyps  = valid["hypothesis"].fillna("").astype(str).tolist()

    # ── BERTSCORE ─────────────────────────────────────────────────────────────
    try:
        _, _, F1 = bert_scorer.score(hyps, refs, verbose=False)
        bertscore_f1 = round(float(F1.mean()), 4)
        print(f"   BERTScore F1       : {bertscore_f1:.4f}")
    except Exception as e:
        bertscore_f1 = float("nan")
        print(f"   BERTScore          : FAILED ({e})")

    # ── SEMANTIC SIMILARITY ───────────────────────────────────────────────────
    try:
        ref_embs = sem_model.encode(refs, batch_size=64, show_progress_bar=False)
        hyp_embs = sem_model.encode(hyps, batch_size=64, show_progress_bar=False)
        sims = [float(cos_sim([r], [h])[0][0]) for r, h in zip(ref_embs, hyp_embs)]
        avg_sem_sim = round(float(np.mean(sims)), 4)
        print(f"   Semantic sim       : {avg_sem_sim:.4f}")
    except Exception as e:
        avg_sem_sim = float("nan")
        print(f"   Semantic sim       : FAILED ({e})")

    rows.append({
        "model"                      : model_name,
        "corpus_wer"                 : float(summary["wer"]),
        "corpus_cer"                 : float(summary["cer"]),
        "corpus_chrf"                : float(summary.get("chrf",                        float("nan"))),
        "bertscore_f1"               : bertscore_f1,
        "avg_semantic_similarity"    : avg_sem_sim,
        "ser"                        : float(summary.get("ser",                         float("nan"))),
        "wer_p50"                    : float(summary.get("wer_p50",                     float("nan"))),
        "wer_p90"                    : float(summary.get("wer_p90",                     float("nan"))),
        "wer_p95"                    : float(summary.get("wer_p95",                     float("nan"))),
        "pct_perfect"                : float(summary.get("pct_perfect",                 float("nan"))),
        "pct_catastrophic"           : float(summary.get("pct_catastrophic",            float("nan"))),
        "sub_rate"                   : float(summary.get("sub_rate",                    float("nan"))),
        "ins_rate"                   : float(summary.get("ins_rate",                    float("nan"))),
        "del_rate"                   : float(summary.get("del_rate",                    float("nan"))),
        "avg_hallucination_ratio"    : float(summary.get("hallucination_ratio",         float("nan"))),
        "avg_script_contamination"   : float(summary.get("script_contamination",        float("nan"))),
        "avg_repetition_score"       : float(summary.get("repetition_score",            float("nan"))),
        "empty_rate"                 : float(summary.get("is_empty",                    float("nan"))),
        "avg_punct_f1"               : float(summary.get("punct_f1",                    float("nan"))),
        "avg_rtf"                    : float(summary["rtf"]),
        "throughput_audio_hrs_per_hr": float(summary.get("throughput_audio_hrs_per_hr", float("nan"))),
        "peak_vram_gb"               : float(summary.get("peak_vram_gb",                float("nan"))),
        "wer_lt5s"                   : float(summary.get("wer_lt5s",                    float("nan"))),
        "wer_5to15s"                 : float(summary.get("wer_5to15s",                  float("nan"))),
        "wer_gt15s"                  : float(summary.get("wer_gt15s",                   float("nan"))),
        "avg_inf_time_sec"           : float(summary["inference_time_sec"]),
        "n_clips"                    : int(len(per_clip)),
    })

# ════════════════════════════════════════════════════════════════
#  MASTER TABLE
# ════════════════════════════════════════════════════════════════
master = (
    pd.DataFrame(rows)
    .sort_values("corpus_wer")
    .reset_index(drop=True)
)
master.index += 1
master.index.name = "rank"

master_path = os.path.join(RESULTS_DIR, "results_master.csv")
master.to_csv(master_path)
print(f"\nMaster results saved → {master_path}\n")

display_cols = [
    "model", "corpus_wer", "corpus_chrf", "bertscore_f1",
    "ser", "ins_rate", "avg_repetition_score",
    "throughput_audio_hrs_per_hr", "peak_vram_gb",
]
print(master[display_cols].to_string())